# Disease ↔ Phenotype Relation-Wise Merge

Merges Disease–Phenotype triples from Monarch and CrossBAR; resolves disease names
via DO/MESH; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_PHENOTYPE/ALL_DISEASE_PHENOTYPE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Disease Lookup Dictionaries

In [2]:
# Disease Ontology
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [3]:
# MedGen: source_id → preferred name; MONDO → MESH CUI
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [4]:
# MESH combined and supplementary: ID → Name
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


## 2. Helper — Resolve Disease Head Node

In [7]:
def resolve_disease_node(df, node):
    """Map MONDO→MESH, derive detail_name, assign id_is for head or tail."""
    df[node] = df[node].map(MONDO_2_MESH_dict).fillna(df[node])
    df = df[~df[node].astype(str).str.startswith('MONDO')].copy()
    df[f'{node}_detail_name'] = df[node].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df[f'{node}_id_is'] = np.where(
        df[node].isna(), None,
        np.where(df[node].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. Monarch

In [8]:
Monarch_Disease_Phenotype = pd.read_csv(PROC_DIR + 'Monarch/Human/Disease_Phenotype_human_MONARCH.csv')
Monarch_Disease_Phenotype.columns = Monarch_Disease_Phenotype.columns.str.lower()
Monarch_Disease_Phenotype.rename(columns={'kgsource': 'kg_source', 'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)
Monarch_Disease_Phenotype = resolve_disease_node(Monarch_Disease_Phenotype, 'head')
print(f"Monarch:  {len(Monarch_Disease_Phenotype):,} rows")

Monarch_Chemical_Pathway['kg_type'] = 'Generalised'
Monarch_Chemical_Pathway['kg_source'] = 'Monarch_KG'
Monarch_Chemical_Pathway['species'] = 'Homo species'

Monarch_Disease_Phenotype.head(2)

Monarch:  258,544 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_orignal,species_species
0,C1332219,Disease_Phenotype,HP:0003581,Disease,has_phenotype,Phenotype,infores:mondo,Monarch_KG,NaN,NaN,MESH,HPO_ID,Adult kidney Wilms tumor,Adult onset,MONDO:0024675,NaN
1,C1332192,Disease_Phenotype,HP:0003581,Disease,has_phenotype,Phenotype,infores:mondo,Monarch_KG,NaN,NaN,MESH,HPO_ID,Adult brain stem neoplasm,Adult onset,MONDO:0024797,NaN


### 3b. CrossBAR

In [11]:
CrossBAR_Disease_Phenotype = pd.read_csv(PROC_DIR + 'crossbar/Disease_Phenotype_Crossbar.csv')
CrossBAR_Disease_Phenotype.columns = CrossBAR_Disease_Phenotype.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Disease_Phenotype):,} rows | columns: {list(CrossBAR_Disease_Phenotype.columns)}")

CrossBAR_Disease_Phenotype['kg_type'] = 'Generalised'
CrossBAR_Disease_Phenotype['species'] = 'Homo species'

CCrossBAR_Disease_PhenotyperossBAR_Disease_Phenotype.head(2)

CrossBAR: 920 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is
0,DOID:423,Disease_Phenotype,HP:0009054,Disease,Phenotype,CROssBAR,EFO:0004145,myopathy,Scapuloperoneal myopathy,DOID,HPO_ID
1,DOID:0080016,Disease_Phenotype,HP:0008517,Disease,Phenotype,CROssBAR,EFO:0003105,spina bifida,Aplasia/Hypoplasia of the sacrum,DOID,HPO_ID


## 3c. Phenknowlator

In [18]:
pheknolator_Disease_Phenotype = pd.read_csv(PROC_DIR + 'pheknowlator/Disease_Phenotype.csv')
pheknolator_Disease_Phenotype.columns = pheknolator_Disease_Phenotype.columns.str.lower()
print(f"CrossBAR: {len(pheknolator_Disease_Phenotype):,} rows | columns: {list(pheknolator_Disease_Phenotype.columns)}")
pheknolator_Disease_Phenotype['kg_type'] = 'Generalised'
pheknolator_Disease_Phenotype['species'] = 'Homo species'

pheknolator_Disease_Phenotype.head(2)

CrossBAR: 63,825 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,DOID:0050886,Disease_Phenotype,HP:0005922,Disease,Phenotype,Troyer syndrome,Abnormal hand morphology,DOID,HPO_ID,pheknowlator,Generalised,Homo species
1,DOID:0070254,Disease_Phenotype,HP:0002079,Disease,Phenotype,congenital disorder of glycosylation type IIb,Hypoplasia of the corpus callosum,DOID,HPO_ID,pheknowlator,Generalised,Homo species


### 3d. GPKG

In [32]:
GPKG = pd.read_csv(PROC_DIR + 'GPKG/Disease_Phenotype_GPKG.csv')
GPKG.columns = GPKG.columns.str.lower()
GPKG.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
GPKG['head_detail_name'] = GPKG['head'].map(DOID_disname_dict)

print(f"GPKG: {len(GPKG):,} rows | columns: {list(GPKG.columns)}")
GPKG['kg_type'] = 'Generalised'
GPKG['species'] = 'Homo species'
GPKG.head(2)

GPKG: 80,347 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_doid', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'head_detail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_doid,tail_detail_name,head_id_is,tail_id_is,head_detail_name,kg_type,species
0,DOID:0070377,Disease_Phenotype,HP:0011097,Disease,described,Phenotype,GPKG,mim:619340,Epileptic spasm,DOID,HP_ID,developmental and epileptic encephalopathy 96,Generalised,Homo species
1,DOID:0070377,Disease_Phenotype,HP:0002187,Disease,described,Phenotype,GPKG,mim:619340,"Intellectual disability, profound",DOID,HP_ID,developmental and epileptic encephalopathy 96,Generalised,Homo species


### 3e. Hetionet

In [33]:
# hetionet = pd.read_csv(PROC_DIR + 'Hetionet/Disease_Symptom_Hetionet.csv')
# hetionet.columns = hetionet.columns.str.lower()
# print(f"Hetionet: {len(hetionet):,} rows | columns: {list(hetionet.columns)}")
# hetionet.head(2)

## 4. Check and Fix Duplicate Columns

In [34]:
SOURCE_DFS = [
    ('Monarch_Disease_Phenotype',  Monarch_Disease_Phenotype),
    ('CrossBAR_Disease_Phenotype', CrossBAR_Disease_Phenotype),
    ('pheknolator_Disease_Phenotype', pheknolator_Disease_Phenotype),
    ('GPKG', GPKG),
    
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Disease_Phenotype] ✓ no duplicates
[CrossBAR_Disease_Phenotype] ✓ no duplicates
[pheknolator_Disease_Phenotype] ✓ no duplicates
[GPKG] ✓ no duplicates


## 5. Align Schemas and Concatenate

In [35]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 403,636 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C1332219,Disease_Phenotype,HP:0003581,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Adult kidney Wilms tumor,Adult onset,NaN,NaN
1,C1332192,Disease_Phenotype,HP:0003581,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Adult brain stem neoplasm,Adult onset,NaN,NaN


In [36]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Disease_Phenotype'}
head_type           : {'Disease'}
tail_type           : {'Phenotype'}
relation_type       : {nan, 'has_mode_of_inheritance', 'has_phenotype', 'described'}
kg_source           : {'Monarch_KG', 'CROssBAR', 'pheknowlator', 'GPKG'}
head_id_is          : {'DOID', 'MESH'}
tail_id_is          : {'HPO_ID', 'HP_ID'}
kg_type             : {nan, 'Generalised'}
species             : {nan, 'Homo species'}


## 6. Standardise Fixed-Value Columns

In [37]:
final_df['head_type'] = 'Disease'
final_df['tail_type'] = 'Phenotype'
final_df['relation']  = 'Disease_Phenotype'
final_df['tail_id_is'] = 'HPO_ID'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Disease_Phenotype'}
Unique head_id_is: {'DOID', 'MESH'}
Unique kg_source: {'Monarch_KG', 'CROssBAR', 'pheknowlator', 'GPKG'}


## 7. Deduplicate and Add Schema Columns

In [38]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
final_df_dedup['head'] = final_df_dedup['head'].astype(str)
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 294,305 rows


## 8. QC — NaN Check

In [42]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,17565,0,17565
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [41]:
final_df_dedup['species'] = 'Homo sapiens'

## 9. Save Output

In [43]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 294,305 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_PHENOTYPE/ALL_DISEASE_PHENOTYPE.csv


# Final Mapping

In [2]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [3]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

final_file

/tmp/ipykernel_302874/3578369478.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  final_file = pd.read_csv(OUT_PATH)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,C0000880,Disease_Phenotype,HP:0000481,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Acanthamoeba keratitis,Abnormal cornea morphology,NaN,Homo sapiens,Homo sapiens,Homo sapiens
1,C0000880,Disease_Phenotype,HP:0000518,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Acanthamoeba keratitis,Cataract,NaN,Homo sapiens,Homo sapiens,Homo sapiens
2,C0000880,Disease_Phenotype,HP:0000593,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Acanthamoeba keratitis,Abnormal anterior chamber morphology,NaN,Homo sapiens,Homo sapiens,Homo sapiens
3,C0000880,Disease_Phenotype,HP:0000613,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Acanthamoeba keratitis,Photophobia,NaN,Homo sapiens,Homo sapiens,Homo sapiens
4,C0000880,Disease_Phenotype,HP:0001089,Disease,has_phenotype,Phenotype,Monarch_KG,MESH,HPO_ID,Acanthamoeba keratitis,Iris atrophy,NaN,Homo sapiens,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294300,DOID:9997,Disease_Phenotype,HP:0030356,Disease,has_phenotype,Phenotype,Monarch_KG,DOID,HPO_ID,peripartum cardiomyopathy,Increased circulating interferon-gamma concent...,NaN,Homo sapiens,Homo sapiens,Homo sapiens
294301,DOID:9997,Disease_Phenotype,HP:0030830,Disease,has_phenotype,Phenotype,Monarch_KG,DOID,HPO_ID,peripartum cardiomyopathy,Crackles,NaN,Homo sapiens,Homo sapiens,Homo sapiens
294302,DOID:9997,Disease_Phenotype,HP:0030848,Disease,has_phenotype,Phenotype,Monarch_KG,DOID,HPO_ID,peripartum cardiomyopathy,Elevated jugular venous pressure,NaN,Homo sapiens,Homo sapiens,Homo sapiens
294303,DOID:9997,Disease_Phenotype,HP:0031295,Disease,has_phenotype,Phenotype,Monarch_KG,DOID,HPO_ID,peripartum cardiomyopathy,Left atrial enlargement,NaN,Homo sapiens,Homo sapiens,Homo sapiens


In [4]:
final_file[final_file['head'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species


In [5]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 294,305 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_PHENOTYPE/ALL_DISEASE_PHENOTYPE.csv
